In [1]:
import torch
import torch.nn as nn

# Check for NVIDIA (Windows/Linux), if not Check for Apple Silicon "Metal Performance Shaders", if not then god help us
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active Device: {device}\n")

Active Device: mps



In [2]:
class CNN(nn.Module):
    

    # WTF is self? Self is just the literal representaton of this network in memory, 
    # Python is annoying once the constuctor finishes execution it deletes all variables it created, 
    # So Self ensure that they belong permanently to this specific class instance.
    
    # constructor
    def __init__(self, num_sensors):
        # Call up the super class "nn.Module" Constructor to inject all the nn infrastructure.
        super(CNN, self).__init__()

        #This produces our Feature Map, just understand what happens when we miss with them, Ref: Phase 4 Arch
        self.conv1 = nn.Conv1d(in_channels=num_sensors, out_channels=32, kernel_size=3)
        
        # Relu(Rectified Linear Unit), Ref: Phase 4 Arch
        self.relu = nn.ReLU()

        # Divide the result from Relu into 2 pairs and pick the largest of them, Large numbers indicate that a major failuire oqquired so we like those 
        self.pool = nn.MaxPool1d(kernel_size=2)

        # CNN Investigation is over now we turn the 3D into 1D to extraxt the features it came up with
        self.flatten = nn.Flatten()

        # Initial length = 30, Conv1d (kernel=3, padding=0): length = 30 - 3 + 1 = 28
        # After MaxPool1d (kernel=2): length = 28 / 2 = 14
        # Flattened size = 32 filters * 14 remaining time steps = 448
        self.fc1 = nn.Linear(32 * 14, 64)
        
        # Single output for binary classification
        # 64 is the number of neurons we have in that layer, each 448 raw clues ar passed to these 64 nodes,
        # Each of these nodes learns a specific relatioship between the failuires we have (ex: heat, pressure for time step 12).
        # Scaling this down risks the model being dumb and unable to find the required relationships between our readings and vice versa
        self.fc2 = nn.Linear(64, 1)


        # Compress the linear output to values between 0 and 1 to extract percentages (ex: 0.2 = 20% probability of failuire due to x)
        self.sigmoid = nn.Sigmoid()

        # The pipleline / Sequence / order of execution
    def forward(self, x):
        # Input 'x' arrives as (Batch, 30, 15)
        # We change the order of the last two columns, 1D convolutions strictly requires the timeline to be the last dimension
        # So result is -> (Batch, 15, 30)
        x = x.permute(0, 2, 1)

        # Order of Extraction
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        #  Order of Classification
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)

        # Binary Probability Output
        x = self.fc2(x)
        x = self.sigmoid(x)

        # This is the predicted failure probability percentage 
        # this output will be used so the network can calculate its loss function and update its weights.
        return x


# Create the Model instance and move it to the device it will run on
model = CNN(num_sensors=15).to(device)

print(model)

CNN(
  (conv1): Conv1d(15, 32, kernel_size=(3,), stride=(1,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=448, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
